<!--BOOK_INFORMATION-->
<img align="left" style="padding-right:10px;" src="figures/PDSH-cover-small.png">
*This notebook contains an excerpt from the [Python Data Science Handbook](http://shop.oreilly.com/product/0636920034919.do) by Jake VanderPlas; the content is available [on GitHub](https://github.com/jakevdp/PythonDataScienceHandbook).*

*The text is released under the [CC-BY-NC-ND license](https://creativecommons.org/licenses/by-nc-nd/3.0/us/legalcode), and code is released under the [MIT license](https://opensource.org/licenses/MIT). If you find this content useful, please consider supporting the work by [buying the book](http://shop.oreilly.com/product/0636920034919.do)!*

<!--NAVIGATION-->
< [Data Indexing and Selection](03.02-Data-Indexing-and-Selection.ipynb) | [Contents](Index.ipynb) | [Handling Missing Data](03.04-Missing-Values.ipynb) >

# 3.4 Operating on Data in Pandas / Pandas 数值运算方法

One of the essential pieces of NumPy is the ability to perform quick element-wise operations, both with basic arithmetic (addition, subtraction, multiplication, etc.) and with more sophisticated operations (trigonometric functions, exponential and logarithmic functions, etc.).
Pandas inherits much of this functionality from NumPy, and the ufuncs that we introduced in [Computation on NumPy Arrays: Universal Functions](02.03-Computation-on-arrays-ufuncs.ipynb) are key to this.

Pandas includes a couple useful twists, however: for unary operations like negation and trigonometric functions, these ufuncs will *preserve index and column labels* in the output, and for binary operations such as addition and multiplication, Pandas will automatically *align indices* when passing the objects to the ufunc.
This means that keeping the context of data and combining data from different sources–both potentially error-prone tasks with raw NumPy arrays–become essentially foolproof ones with Pandas.
We will additionally see that there are well-defined operations between one-dimensional ``Series`` structures and two-dimensional ``DataFrame`` structures.

🐍 一元运算（函数和三角函数），通用函数将在输出结果中保留索引和列标签

🐍 二元运算（加法和乘法），Pandas在传递通用函数时会自动对齐索引进行计算 

## 3.4.1 Ufuncs: Index Preservation / 通用函数：保留索引

Because Pandas is designed to work with NumPy, any NumPy ufunc will work on Pandas ``Series`` and ``DataFrame`` objects.
Let's start by defining a simple ``Series`` and ``DataFrame`` on which to demonstrate this:

In [1]:
import pandas as pd
import numpy as np

In [56]:
# 🐍 RandomState()函数?

rng = np.random.RandomState(42)
ser = pd.Series(rng.randint(0, 10, 4))
ser

0    6
1    3
2    7
3    4
dtype: int32

In [57]:
# 🐍 randit()函数

df = pd.DataFrame(rng.randint(0, 10, (3, 4)),
                  columns=['A', 'B', 'C', 'D'])
df

,A,B,C,D
0,6,9,2,6
1,7,4,3,7
2,7,2,5,4


If we apply a NumPy ufunc on either of these objects, the result will be another Pandas object *with the indices preserved:*

In [5]:
# 🐍 一元函数exp()对Series对象进行运算，结果为保留索引的Series对象

print(ser)
np.exp(ser)

0    6
1    3
2    7
3    4
dtype: int32


0     403.428793
1      20.085537
2    1096.633158
3      54.598150
dtype: float64

Or, for a slightly more complex calculation:

In [58]:
# 🐍 一元函数sin()对DataFrame对象进行运算，结果为保留索引的DataFrame对象

np.sin(df * np.pi / 4)

,A,B,C,D
0,-1.000000,7.071068e-01,1.000000,-1.000000e+00
1,-0.707107,1.224647e-16,0.707107,-7.071068e-01
2,-0.707107,1.000000e+00,-0.707107,1.224647e-16


Any of the ufuncs discussed in [Computation on NumPy Arrays: Universal Functions](02.03-Computation-on-arrays-ufuncs.ipynb) can be used in a similar manner.

## 3.4.2 UFuncs: Index Alignment / 通用函数：对齐索引

For binary operations on two ``Series`` or ``DataFrame`` objects, Pandas will align indices in the process of performing the operation.
This is very convenient when working with incomplete data, as we'll see in some of the examples that follow.

🐍 对Series和DataFrame对象进行二元计算时，Pandas会在计算过程中对齐两个对象的索引

🐍 数据缺失会以NaN填充

### 1. Index alignment in Series / Series 索引对齐

As an example, suppose we are combining two different data sources, and find only the top three US states by *area* and the top three US states by *population*:

In [59]:
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,
                  'California': 423967}, name='area')
population = pd.Series({'California': 38332521, 'Texas': 26448193,
                        'New York': 19651127}, name='population')

In [7]:
area

Alaska        1723337
Texas          695662
California     423967
Name: area, dtype: int64

In [8]:
population

California    38332521
Texas         26448193
New York      19651127
Name: population, dtype: int64

Let's see what happens when we divide these to compute the population density:

In [9]:
population / area

Alaska              NaN
California    90.413926
New York            NaN
Texas         38.018740
dtype: float64

The resulting array contains the *union* of indices of the two input arrays, which could be determined using standard Python set arithmetic on these indices:

🐍 数组的索引是两个输入数组索引的并集

In [15]:
# 🐍 并集 |

# area.index | population.index # | 将对象进行按位或运算，结果为两个对象的并集，字符串对象不支持按位或操作

area.index.union(population.index)

Index(['Alaska', 'California', 'New York', 'Texas'], dtype='object')

Any item for which one or the other does not have an entry is marked with ``NaN``, or "Not a Number," which is how Pandas marks missing data (see further discussion of missing data in [Handling Missing Data](03.04-Missing-Values.ipynb)).
This index matching is implemented this way for any of Python's built-in arithmetic expressions; any missing values are filled in with NaN by default:

In [61]:
# 🐍 并集 A + B

A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])

print(A)
print(B)

A + B

0    2
1    4
2    6
dtype: int64
1    1
2    3
3    5
dtype: int64


0    NaN
1    5.0
2    9.0
3    NaN
dtype: float64

If using NaN values is not the desired behavior, the fill value can be modified using appropriate object methods in place of the operators.
For example, calling ``A.add(B)`` is equivalent to calling ``A + B``, but allows optional explicit specification of the fill value for any elements in ``A`` or ``B`` that might be missing:

In [63]:
# 🐍 并集 A.add(B) 等价于 A + B 
# 🐍 add()函数中，参数 fill_value 自定义缺省值

A.add(B, fill_value=100)

0    102.0
1      5.0
2      9.0
3    105.0
dtype: float64

### 2. Index alignment in DataFrame / DataFrame 索引对齐

A similar type of alignment takes place for *both* columns and indices when performing operations on ``DataFrame``s:

In [64]:
A = pd.DataFrame(rng.randint(0, 10, (2, 2)),
                 columns=list('ab'))
A

,a,b
0,1,7
1,5,1


In [65]:
B = pd.DataFrame(rng.randint(0, 10, (3, 3)),
                 columns=list('bac'))
B

,b,a,c
0,4,0,9
1,5,8,0
2,9,2,6


In [66]:
# 🐍 两个对象的行索引是不同顺序，结果的索引会自动排序

A + B

,a,b,c
0,1.0,11.0,NaN
1,13.0,6.0,NaN
2,NaN,NaN,NaN


In [67]:
A.add(B, fill_value=0)

,a,b,c
0,1.0,11.0,9.0
1,13.0,6.0,0.0
2,2.0,9.0,6.0


Notice that indices are aligned correctly irrespective of their order in the two objects, and indices in the result are sorted.
As was the case with ``Series``, we can use the associated object's arithmetic method and pass any desired ``fill_value`` to be used in place of missing entries.
Here we'll fill with the mean of all values in ``A`` (computed by first stacking the rows of ``A``):

In [68]:
A = pd.DataFrame(rng.randint(0, 10, (2, 2)),
                 columns=list('ab'))

A

,a,b
0,3,8
1,2,4


In [69]:
A['b'][1]

4

In [72]:
# 🐍 stack()函数将二维数组压缩为一维数组

print(type(A))
fill = A.stack()

fill
# print(type(fill))

<class 'pandas.core.frame.DataFrame'>


0  a    3
   b    8
1  a    2
   b    4
dtype: int32

In [54]:
fill[0, 'b']

8

In [35]:
fill.mean()

4.25

In [73]:
# 🐍 stack()函数将二维数组压缩为一维数组

fill = A.stack().mean()
A.add(B, fill_value=fill)

,a,b,c
0,3.00,12.00,13.25
1,10.00,9.00,4.25
2,6.25,13.25,10.25


The following table lists Python operators and their equivalent Pandas object methods:

| Python Operator | Pandas Method(s)                      |
|-----------------|---------------------------------------|
| ``+``           | ``add()``                             |
| ``-``           | ``sub()``, ``subtract()``             |
| ``*``           | ``mul()``, ``multiply()``             |
| ``/``           | ``truediv()``, ``div()``, ``divide()``|
| ``//``          | ``floordiv()``                        |
| ``%``           | ``mod()``                             |
| ``**``          | ``pow()``                             |


## 3.4.3 Ufuncs: Operations Between DataFrame and Series

## 通用函数：DataFrame 与 Series 的运算

When performing operations between a ``DataFrame`` and a ``Series``, the index and column alignment is similarly maintained.
Operations between a ``DataFrame`` and a ``Series`` are similar to operations between a two-dimensional and one-dimensional NumPy array.
Consider one common operation, where we find the difference of a two-dimensional array and one of its rows:

In [78]:
A = rng.randint(10, size=(3, 4))
B = pd.DataFrame(A)
type(B)

pandas.core.frame.DataFrame

In [87]:
# 🐍 让一个二维数组减去自身的一行数据：Pandas默认按行计算

B - B.iloc[0]

SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='? (1699891015.py, line 3)

According to NumPy's broadcasting rules (see [Computation on Arrays: Broadcasting](02.05-Computation-on-arrays-broadcasting.ipynb)), subtraction between a two-dimensional array and one of its rows is applied row-wise.

In Pandas, the convention similarly operates row-wise by default:

In [84]:
df = pd.DataFrame(A, columns=list('QRST'))
df

,Q,R,S,T
0,8,7,4,1
1,4,7,9,8
2,8,0,8,6


In [92]:
# 🐍 默认按行计算

df - df.iloc[0]

,Q,R,S,T
0,0,0,0,0
1,-4,0,5,7
2,0,-7,4,5


If you would instead like to operate column-wise, you can use the object methods mentioned earlier, while specifying the ``axis`` keyword:

In [41]:
# 🐍 按列计算：参数 axis 设置--> 0为列， 1为行

df.subtract(df['R'], axis=0)

,Q,R,S,T
0,2,0,3,1
1,4,0,-2,2
2,-4,0,-6,-4


Note that these ``DataFrame``/``Series`` operations, like the operations discussed above, will automatically align  indices between the two elements:

In [42]:
# 🐍 iloc()隐式索引，第0行，每隔2列

halfrow = df.iloc[0, ::2]
halfrow

Q    2
S    3
Name: 0, dtype: int32

In [43]:
# 🐍 Pandas在运算时保存数据内容，避免数据类型差异和维度问题

df - halfrow

,Q,R,S,T
0,0.0,NaN,0.0,NaN
1,5.0,NaN,-2.0,NaN
2,3.0,NaN,0.0,NaN


This preservation and alignment of indices and columns means that operations on data in Pandas will always maintain the data context, which prevents the types of silly errors that might come up when working with heterogeneous and/or misaligned data in raw NumPy arrays.

<!--NAVIGATION-->
< [Data Indexing and Selection](03.02-Data-Indexing-and-Selection.ipynb) | [Contents](Index.ipynb) | [Handling Missing Data](03.04-Missing-Values.ipynb) >